In [2]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine

db_path = str("./db/Chinook.db")
db_path = f"sqlite:///{db_path}"
engine = create_engine(db_path)
db = SQLDatabase(engine=engine)

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai.chat_models import AzureChatOpenAI

load_dotenv()
api_key = os.environ['api_key']
azure_endpoint = os.environ['azure_endpoint']
api_version = os.environ['api_version']

llm = AzureChatOpenAI(
    api_version = api_version,
    azure_endpoint = azure_endpoint,
    azure_deployment = "gpt-4o",
    model_name = "gpt-4o",
    api_key = api_key,
    temperature = 1.0,
)

In [4]:
from openinference.instrumentation.langchain import LangChainInstrumentor
from phoenix.otel import register

# configure the Phoenix tracer
tracer_provider = register(
  project_name="Multi-Agent System",
  auto_instrument=True
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: Multi-Agent System
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
from sqlalchemy import inspect

def get_database_schema():
    inspector = inspect(engine)
    schema = ""
    for table_name in inspector.get_table_names():
        schema += f"Table: {table_name}\n"
        for column in inspector.get_columns(table_name):
            col_name = column["name"]
            col_type = str(column["type"])
            if column.get("primary_key"):
                col_type += ", Primary Key"
            if column.get("foreign_keys"):
                fk = list(column["foreign_keys"])[0]
                col_type += f", Foreign Key to {fk.column.table.name}.{fk.column.name}"
            schema += f"- {col_name}: {col_type}\n"
        schema += "\n"
    return schema

In [6]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# Define the tools for the agent to use
@tool
def check_relevance(question: str) -> str:
    """Determines whether the given question is relevant to the schema.

    Args:
        question: The question to check.

    Returns:
        str: The relevance of the question ("relevant" or "not_relevant").

    Raises:
        Exception: If there is an error determining relevance.
    """
    try:
        question = question
        schema = get_database_schema()
        system = """You are an assistant that determines whether a given question is related to the following database schema.

Schema:
{schema}

Respond with only "relevant" or "not_relevant".
""".format(schema=schema)
        human = f"Question: {question}"
        check_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system),
                ("human", human),
            ]
        )
        # Use llm to process the check
        relevance_checker = check_prompt | llm
        relevance = relevance_checker.invoke({})
        return relevance.content
    
    except Exception as e:
        raise Exception(f"Error checking relevance: {str(e)}")

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# Define the tools for the agent to use
@tool
def convert_nl_to_sql(question: str) -> str:
    """Converts a natural language question into an SQL query based on the given schema.

    Args:
        question: The question to convert into an SQL query.

    Returns:
        str: The generated SQL query.
    
    Raises:
        Exception: If there is an error generating the SQL query.
    """
    try:
        schema = get_database_schema()
        system = """You are an assistant that converts natural language questions into SQL queries based on the following schema:

{schema}

Provide only the SQL query without any explanations. Alias columns appropriately to match the expected keys in the result.

For example, alias 'food.name' as 'food_name' and 'food.price' as 'price'.
""".format(schema=schema)
        
        convert_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system),
                ("human", f"Question: {question}"),
            ]
        )
        # Use llm to process the conversion
        sql_generator = convert_prompt | llm
        result = sql_generator.invoke({"question": question})
        return result.content.replace('```sql\n', '').replace('\n```', '')
    
    except Exception as e:
        raise Exception(f"Error converting natural language to SQL: {str(e)}")


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
import sqlparse

@tool
def query_checker(sql_query: str) -> str:
    """Checks and corrects SQL queries before execution.

    Args:
        sql_query: The SQL query to validate and possibly correct.

    Returns:
        str: The original or corrected SQL query.
    
    Raises:
        Exception: If there is an error correcting the SQL query.
    """
    sql_query = sql_query.strip()
    
    try:
        parsed = sqlparse.parse(sql_query)
        
        if not parsed:
            raise ValueError("Empty or invalid SQL query")

        system_message = """You are an assistant that checks SQL queries for correctness and corrects any syntax errors. 
        If the query is valid, return the same query. If it has errors, correct them and return the corrected query.
        
Query:
{sql_query}
        
Return only the corrected SQL query, if needed. If the query is valid, return the same query.
""".format(sql_query=sql_query)
        
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system_message),
                ("human", f"SQL Query: {sql_query}")
            ]
        )
        # Use llm to verify and correct the SQL query
        query_corrector = prompt | llm  
        corrected_query = query_corrector.invoke({"sql_query": sql_query})

        if corrected_query.content.strip() != sql_query:
            return corrected_query.content.strip()
        
        return sql_query
    
    except Exception as e:
        raise Exception(f"Error checking or correcting the SQL query: {str(e)}")


In [9]:
from langchain_core.tools import tool
from sqlalchemy import text

# Define the tools for the agent to use
@tool
def execute_sql(sql_query: str) -> str:
    """Executes the given SQL query and returns the result as a string.

    Args:
        sql_query: The SQL query to execute.

    Returns:
        str: The result of the SQL query execution, either data or error message.
    """
    sql_query = sql_query.strip()
    
    try:
        result = db.run(text(sql_query))
        return result
    
    except Exception as e:
        result_str = f"Error executing SQL query: {str(e)}"
        print(f"Error executing SQL query: {str(e)}")
    
    # return result_str
    return result


In [10]:
tools1 = [convert_nl_to_sql, check_relevance, query_checker, execute_sql]
llm_with_tools1 = llm.bind_tools(tools1)

In [11]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from typing import List, Any

class GraphState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [12]:
# Node
def sqlagent(state: GraphState):
    response = llm_with_tools1.invoke(state["messages"])
    return {"messages": [response]}


In [13]:
from langgraph.prebuilt import ToolNode

relevance_tool = ToolNode([check_relevance])
conversion_tool = ToolNode([convert_nl_to_sql])
correct_tool = ToolNode([query_checker])
execution_tool = ToolNode([execute_sql])

In [14]:
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display

# Define the routing functions
def route_to_tools(state: GraphState):
    messages = state["messages"]
    last_message = messages[-1]
    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return END
    
    tool_name = last_message.tool_calls[0]["name"]
    
    if tool_name == "check_relevance":
        return "check_relevance"
    elif tool_name == "convert_nl_to_sql":
        return "convert_nl_to_sql"
    elif tool_name == "query_checker":
        return "query_checker"
    elif tool_name == "execute_sql":
        return "execute_sql"
    else:
        return END


# Define a new graph for the agent
builder = StateGraph(GraphState)
builder.add_node("SQL_Agent", sqlagent)
builder.add_node("check_relevance", relevance_tool)
builder.add_node("convert_nl_to_sql", conversion_tool)
builder.add_node("query_checker", correct_tool)
builder.add_node("execute_sql", execution_tool)

builder.add_conditional_edges("SQL_Agent", route_to_tools, ["check_relevance", "convert_nl_to_sql", "query_checker", "execute_sql", END])
builder.add_edge("check_relevance", "SQL_Agent")
builder.add_edge("convert_nl_to_sql", "SQL_Agent")
builder.add_edge("query_checker", "SQL_Agent")
builder.add_edge("execute_sql", "SQL_Agent")
builder.add_edge("SQL_Agent", END)

builder.set_entry_point("SQL_Agent")

sql_agent = builder.compile(name="SQL_Agent")
display(Image(sql_agent.get_graph(xray=True).draw_png()))

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [14]:
# from langchain_core.messages import HumanMessage
# messages = [HumanMessage(content="Which country's customer spent the most?")]
# result = sql_agent.invoke({"messages": messages})

In [15]:
# result["messages"]

In [15]:
import os
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import AzureOpenAIEmbeddings
from langchain_core.tools import tool

@tool
def retrieve_context(query: str):
    """Search for relevant documents in the local directory."""
    
    # Get a list of all text files in the current directory
    path ="./documents"
    filenames = [f for f in os.listdir(path) if f.endswith('.txt')]  # Assuming you're working with .txt files
    
    docs_list = []
    
    # Load documents from the local text files
    for filename in filenames:
        try:
            file_path = os.path.join(path, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
                doc = Document(page_content=content, metadata={"source": filename})
                docs_list.append(doc)
        except Exception as e:
            print(f"Error reading file {filename}: {e}")
    
    # Split the documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    doc_splits = text_splitter.split_documents(docs_list)
    
    # Create Azure OpenAI embeddings
    embeddings = AzureOpenAIEmbeddings(
        azure_endpoint=azure_endpoint,
        api_key=api_key,
        api_version=api_version,
    )

    # Create the vector store
    vectorstore = Chroma.from_documents(
        documents=doc_splits,
        collection_name="local_docs",
        embedding=embeddings,
    )
    
    # Get the retriever and retrieve relevant documents
    retriever = vectorstore.as_retriever()
    results = retriever.invoke(query)

    # Return the content of the retrieved documents
    return "\n".join([doc.page_content for doc in results])


In [16]:
tools2 = [retrieve_context]
llm_with_tools2 = llm.bind_tools(tools2)

In [17]:
def ragagent(state: GraphState):
    system_message = """You are a specialized Retrieval-Augmented Generation (RAG) agent. Your primary function is to perform knowledge retrieval and generate responses using the tools assigned to you.
    You must utilize the provided tools for all tasks and queries. 
    Do not retry the tool more than 3 times if answer is not generated.
    Do not rely on your own knowledge to generate answer. 
    Your responses must solely reflect the findings obtained through the tool invocation.
    If provided tools do not give suitable answer then inform the supervisor.
    """
    from langchain_core.prompts import ChatPromptTemplate
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_message),
            ("human", "{input}"),
        ]
    )
    chain = prompt | llm_with_tools2
    response = chain.invoke({"input": state["messages"]})
    return {"messages": [response]}

In [18]:
from langgraph.prebuilt import ToolNode

retriever_tool = ToolNode([retrieve_context])

In [19]:
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display

# Define the routing functions
def route_to_tools(state: GraphState):
    messages = state["messages"]
    last_message = messages[-1]
    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return END
    
    tool_name = last_message.tool_calls[0]["name"]
    
    if tool_name == "retrieve_context":
        return "retrieve_context"
    else:
        return END
    
# Define a new graph for the agent
builder = StateGraph(GraphState)
builder.add_node("RAG_Agent", ragagent)
builder.add_node("retrieve_context", retriever_tool)

builder.add_edge(START, "RAG_Agent")
builder.add_conditional_edges("RAG_Agent", route_to_tools, ["retrieve_context", END])
builder.add_edge("retrieve_context", "RAG_Agent")

rag_agent = builder.compile(name="RAG_Agent")
display(Image(rag_agent.get_graph(xray=True).draw_png()))


ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [20]:
# from langchain_core.messages import HumanMessage

# messages = [HumanMessage(content="What does Lilian Weng say about the types of agent memory?")]
# result = rag_agent.invoke({"messages": messages})

In [21]:
# result["messages"]

In [22]:
import requests
from langchain_core.tools import tool

@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Fetches the current weather for a given latitude and longitude.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.

    Returns:
        str: A message about the current weather at the given latitude and longitude.
    """
    try:
        # API key should be securely managed, not hardcoded.
        api_key = "6481e51a5a5af10f10f706f4e57d36d2"
        api_url = f"https://api.openweathermap.org/data/2.5/weather?lat={latitude}&lon={longitude}&appid={api_key}&units=metric"
        response = requests.get(api_url)
        response.raise_for_status()
        weather_data = response.json()

        if weather_data.get('cod') == 200:
            main = weather_data.get('main', {})
            weather = weather_data.get('weather', [{}])[0]
            
            temperature = main.get('temp')
            weather_description = weather.get('description', 'no description available')
            
            return f"The weather at latitude {latitude} and longitude {longitude} is currently {weather_description} with a temperature of {temperature}°C."
        else:
            return f"Sorry, I couldn't fetch the weather data for the given coordinates. Please check the coordinates."

    except requests.exceptions.RequestException as e:
        return f"Error fetching weather data: {str(e)}"

    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


In [23]:
tools3 = [get_weather]
llm_with_tools3 = llm.bind_tools(tools3)

In [24]:
def apiagent(state: GraphState):
    response = llm_with_tools3.invoke(state["messages"])
    return {"messages": [response]}


In [25]:
from langgraph.prebuilt import ToolNode

weather_tool = ToolNode([get_weather])

In [26]:
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display

# Define the routing functions
def route_to_tools(state: GraphState):
    messages = state["messages"]
    last_message = messages[-1]
    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return END
    
    tool_name = last_message.tool_calls[0]["name"]
    
    if tool_name == "get_weather":
        return "get_weather"
    else:
        return END
    
# Define a new graph for the agent
builder = StateGraph(GraphState)
builder.add_node("API_Agent", apiagent)
builder.add_node("get_weather", weather_tool)

builder.add_edge(START, "API_Agent")
builder.add_conditional_edges("API_Agent", route_to_tools, ["get_weather", END])
builder.add_edge("get_weather", "API_Agent")
builder.add_edge("API_Agent", END)

api_agent = builder.compile(name="API_Agent")
display(Image(api_agent.get_graph(xray=True).draw_png()))


ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [27]:
# from langchain_core.messages import HumanMessage

# messages = [HumanMessage(content="Tell me the weather at latitude 44.34 and longitude 10.99")]
# result = api_agent.invoke({"messages": messages})

In [28]:
# result["messages"]

In [29]:
# Supervisor
from langgraph_supervisor import create_supervisor

prompt = (
    "You are a team supervisor overseeing three specialized agents: a SQL agent, a RAG (Retrieval-Augmented Generation) agent, and an API agent."
    "Utilize the SQL agent strictly for executing SQL queries and performing database operations."
    "Employ the RAG agent solely for tasks related to knowledge retrieval."
    "Engage the API agent exclusively for making API calls or integrating with external services."
    "As the supervisor, your role is to assess incoming requests and direct them to the appropriate agent based on the task requirements. If a task falls outside the capabilities of any of the agents, you should respond using your own knowledge."
    "Ensure that the results returned from each agent are delivered as-is, without any alterations, to maintain the integrity of the information provided. Uphold a clear separation of responsibilities for each agent, guaranteeing that the correct agent is assigned to each specific task."
)

# Create supervisor workflow
workflow = create_supervisor(
    [sql_agent, rag_agent, api_agent],
    model=llm,
    prompt=prompt,
    output_mode="full_history",
)


app = workflow.compile(name="Supervisor")
display(Image(app.get_graph(xray=True).draw_png()))

ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

In [31]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Which country's customer spent the most?"),]
result = app.invoke({"messages": messages})

In [32]:
result["messages"]

[HumanMessage(content="Which country's customer spent the most?", additional_kwargs={}, response_metadata={}, id='0dfe0bd0-112e-4c18-99a6-c9b46538dd00'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_umlJuv2lZyJtEZRBqEeqRn2W', 'function': {'arguments': '{}', 'name': 'transfer_to_sql_agent'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 246, 'total_tokens': 259, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BXOuy62l9AYP4d96FJTaLHQ8ty4oX', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'sever

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="What does Lilian Weng say about the types of agent memory?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Describe Long Term Memory")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [35]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Tell me the weather at latitude 44.34 and longitude 10.99")]
result = app.invoke({"messages": messages})

In [36]:
result["messages"]

[HumanMessage(content='Tell me the weather at latitude 44.34 and longitude 10.99', additional_kwargs={}, response_metadata={}, id='f9116aed-20d9-4c7f-bd60-8004e761f0a6'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_MsNbXaOyP1s2PIWOO4ICA4RL', 'function': {'arguments': '{}', 'name': 'transfer_to_api_agent'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 255, 'total_tokens': 268, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BXOw95iMg7lX5DB98pP4bvMk5y6gi', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtere

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="What about latitude 18.52 and longitude 73.85")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="How many customers are there from each country?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="What was the first question?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="How many customers are from Argentina?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="How many customers are there from each country?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="What does Lilian Weng say about the types of agent memory?")]
result = app.invoke({"messages": messages})

In [ ]:
result["messages"]